In [0]:
# Section 4 - Create a Notebook and Read the CSV Datasets

catalog = "workspace"
schema = "retail_lab"
base = f"/Volumes/{catalog}/{schema}/raw_files"

customers = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/customers.csv"))

products = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/products.csv"))

orders = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base}/orders.csv"))

display(customers.limit(5))
display(products.limit(5))
display(orders.limit(5))

customer_id,customer_name,email,city,region,signup_date,customer_segment
C0001,Customer_001,customer1@example.com,Guwahati,East,2023-03-31,Small Business
C0002,Customer_002,customer2@example.com,Pune,West,2025-02-12,Small Business
C0003,Customer_003,customer3@example.com,Kochi,South,2024-10-16,Small Business
C0004,Customer_004,customer4@example.com,Surat,West,2024-03-14,Corporate
C0005,Customer_005,customer5@example.com,Nagpur,Central,2024-03-09,Corporate


product_id,product_name,category,unit_cost,list_price
P001,Wireless Mouse,Electronics,8047.36,8559.07
P002,Keyboard,Electronics,6817.61,13805.23
P003,Monitor,Electronics,7645.84,10031.39
P004,Headphones,Electronics,6745.75,5575.51
P005,Webcam,Electronics,447.35,7945.41


order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery


In [0]:
# Section 5: Convert CSV to Parquet
# Ensure the 'parquet' volume exists inside retail_lab
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.parquet")

parquet_base = f"/Volumes/{catalog}/{schema}/parquet"

# Write the three DataFrames as Parquet datasets
customers.write.mode("overwrite").parquet(f"{parquet_base}/customers")
products.write.mode("overwrite").parquet(f"{parquet_base}/products")
orders.write.mode("overwrite").parquet(f"{parquet_base}/orders")

# Read the Parquet data back and inspect its schema and first 10 rows
orders_parquet = spark.read.parquet(f"{parquet_base}/orders")
orders_parquet.printSchema()
display(orders_parquet.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- payment_method: string (nullable = true)



order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking


In [0]:
# Section 6: Register Temporary Views for SQL

# This loads the three Parquet datasets and registers them as temporary SQL views named customers, products, and orders so you can query them using SQL (%sql) or PySpark.

customers_parquet = spark.read.parquet(f"{parquet_base}/customers")
products_parquet = spark.read.parquet(f"{parquet_base}/products")
orders_parquet = spark.read.parquet(f"{parquet_base}/orders")

customers_parquet.createOrReplaceTempView("customers")
products_parquet.createOrReplaceTempView("products")
orders_parquet.createOrReplaceTempView("orders")

In [0]:
%sql
-- Section 7 - Level 1: Basics, Questions 7-12
-- Question 7: Display the first 10 orders
-- Following the workbook's suggested workflow, we'll use SQL (%sql at the top of a cell) for Level 1, along with PySpark for inspecting schemas in Question 8.

SELECT * FROM orders LIMIT 10;

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking


In [0]:
# Section 7 - Level 1
# Question 8: Show the schema and row count of each dataset (PySpark)
for name, df in [("customers", customers_parquet), ("products", products_parquet), ("orders", orders_parquet)]:
    print(f"=== {name} (Row count: {df.count()}) ===")
    df.printSchema()

=== customers (Row count: 120) ===
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

=== products (Row count: 21) ===
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- list_price: double (nullable = true)

=== orders (Row count: 1500) ===
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- p

In [0]:
%sql
-- Section 7 - Level 1
-- Question 9: Find all orders with status Delivered
SELECT * FROM orders WHERE order_status = 'Delivered';

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00011,2025-03-30,C0002,P019,4,1714.55,0.0,Delivered,2025-04-02,Credit Card
O00012,2025-07-27,C0015,P006,2,7442.24,0.1,Delivered,2025-08-02,UPI
O00013,2026-01-22,C0074,P020,7,17319.87,0.0,Delivered,2026-01-24,Net Banking
O00014,2024-02-08,C0038,P018,4,11400.54,0.15,Delivered,2024-02-15,Debit Card


In [0]:
%sql
-- Section 7 - Level 1
-- Question 10: Show orders placed in 2025
SELECT * FROM orders WHERE YEAR(TO_DATE(order_date)) = 2025;

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00011,2025-03-30,C0002,P019,4,1714.55,0.0,Delivered,2025-04-02,Credit Card
O00012,2025-07-27,C0015,P006,2,7442.24,0.1,Delivered,2025-08-02,UPI
O00033,2025-01-05,C0018,P005,6,7945.41,0.0,Shipped,2025-01-08,Debit Card
O00034,2025-09-11,C0020,P004,2,5575.51,0.1,Shipped,2025-09-13,Net Banking
O00037,2025-06-06,C0065,P001,4,8559.07,0.0,Delivered,2025-06-13,Net Banking
O00039,2025-01-21,C0012,P018,1,11400.54,0.0,Delivered,2025-01-24,UPI
O00041,2025-10-11,C0078,P013,6,1757.35,0.15,Delivered,2025-10-12,Cash on Delivery


In [0]:
%sql
-- Section 7 - Level 1
-- Question 11: List all distinct order statuses
SELECT DISTINCT order_status FROM orders ORDER BY order_status;

order_status
Cancelled
Delivered
Pending
Returned
Shipped


In [0]:
%sql
-- Section 7 - Level 1
-- Question 12: Find orders with a missing shipping_date
SELECT * FROM orders WHERE shipping_date IS NULL;

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking
O00019,2026-02-11,C0114,P006,1,7442.24,0.0,Pending,null,Net Banking
O00028,2026-02-03,C0097,P010,4,14142.58,0.1,Delivered,null,UPI
O00035,2026-01-03,C0007,P005,2,7945.41,0.15,Cancelled,null,Debit Card
O00040,2026-08-15,C0097,P019,2,1714.55,0.2,Pending,null,Cash on Delivery
O00049,2025-06-25,C0113,P016,1,12579.32,0.2,Pending,null,Debit Card
O00055,2026-03-12,C0089,P019,6,1714.55,0.15,Cancelled,null,Debit Card


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 13: Calculate the total number of orders by status
SELECT order_status, COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;

order_status,order_count
Delivered,938
Shipped,228
Pending,138
Cancelled,114
Returned,82


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 14: Calculate total quantity sold by product
SELECT p.product_name, SUM(o.quantity) AS units_sold
FROM orders o
JOIN products p USING (product_id)
GROUP BY p.product_name
ORDER BY units_sold DESC;

product_name,units_sold
Monitor,359
Filing Cabinet,358
Power Bank,348
Markers,331
Wireless Mouse,299
Standing Desk,299
Notebook,292
USB Hub,290
Printer Paper,289
Desk,287


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 15: Calculate revenue by payment method.
-- Revenue = quantity * unit_price * (1 - discount_pct). Treat a missing discount as 0.
--
-- Note on COALESCE(discount_pct, 0):
-- COALESCE checks its arguments from left to right and returns the first non-NULL value.
-- - If discount_pct has a value (e.g., 0.10), it returns 0.10.
-- - If discount_pct is NULL (missing), it replaces it with 0.
-- Without COALESCE, any math with NULL becomes NULL (e.g., 100 * (1 - NULL) = NULL),
-- which would cause orders with missing discounts to be excluded from the revenue sum.

SELECT payment_method,
       ROUND(SUM(quantity * unit_price * (1 - COALESCE(discount_pct, 0))), 2) AS revenue
FROM orders
GROUP BY payment_method
ORDER BY revenue DESC;

payment_method,revenue
Cash on Delivery,1.049678575E7
Net Banking,9721743.59
Credit Card,9285677.68
Debit Card,8893619.04
UPI,8409202.55


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 16: Find monthly revenue for 2025
--
-- Note: DATE_FORMAT(TO_DATE(order_date), 'yyyy-MM') formats the date as Year-Month (e.g., '2025-01')
-- so we can group and sort orders month by month.

SELECT DATE_FORMAT(TO_DATE(order_date), 'yyyy-MM') AS order_month,
       ROUND(SUM(quantity * unit_price * (1 - COALESCE(discount_pct, 0))), 2) AS revenue
FROM orders
WHERE YEAR(TO_DATE(order_date)) = 2025
GROUP BY DATE_FORMAT(TO_DATE(order_date), 'yyyy-MM')
ORDER BY order_month;

order_month,revenue
2025-01,1326127.75
2025-02,1012008.91
2025-03,1840762.33
2025-04,1373788.27
2025-05,1371503.45
2025-06,1625915.36
2025-07,1357006.79
2025-08,1427293.05
2025-09,1429870.85
2025-10,1349646.6


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 17: Find the top 5 customers by total order value
--
-- Note: We JOIN orders and customers on customer_id using USING (customer_id),
-- calculate the total order value per customer, sort from highest to lowest (DESC),
-- and take the top 5 rows with LIMIT 5.

SELECT c.customer_id, c.customer_name,
       ROUND(SUM(o.quantity * o.unit_price * (1 - COALESCE(o.discount_pct, 0))), 2) AS total_value
FROM orders o
JOIN customers c USING (customer_id)
GROUP BY c.customer_id, c.customer_name
ORDER BY total_value DESC
LIMIT 5;

customer_id,customer_name,total_value
C0074,Customer_074,874382.82
C0019,Customer_019,721784.82
C0029,Customer_029,677281.03
C0018,Customer_018,624319.59
C0013,Customer_013,617114.3


In [0]:
%sql
-- Section 7 - Level 2: Aggregations
-- Question 18: Calculate average order value by customer segment
--
-- Explanation:
-- 1. WITH order_values AS (...): Creates a temporary step (CTE) that calculates the total value
--    of each individual order (grouped by order_id and customer_segment).
-- 2. Main SELECT: Takes the average (AVG) of those order values for each customer_segment
--    and rounds it to 2 decimal places using ROUND(..., 2).

WITH order_values AS (
  SELECT o.order_id, c.customer_segment,
         SUM(o.quantity * o.unit_price * (1 - COALESCE(o.discount_pct, 0))) AS order_value
  FROM orders o
  JOIN customers c USING (customer_id)
  GROUP BY o.order_id, c.customer_segment
)
SELECT customer_segment, ROUND(AVG(order_value), 2) AS avg_order_value
FROM order_values
GROUP BY customer_segment;

customer_segment,avg_order_value
Consumer,30727.98
Small Business,30208.47
Corporate,32854.83


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 19: Join orders to customers and show order ID, customer name, city and region (PySpark)
#
# Explanation:
# .join(customers_parquet, "customer_id", "inner") performs an inner join on customer_id,
# and .select(...) picks only the 4 requested columns.

orders_with_customers = orders_parquet.join(customers_parquet, "customer_id", "inner")
display(orders_with_customers.select("order_id", "customer_name", "city", "region"))

order_id,customer_name,city,region
O00001,Customer_032,Delhi,North
O00002,Customer_089,Guwahati,East
O00003,Customer_087,Hyderabad,South
O00004,Customer_092,Delhi,North
O00005,Customer_078,Raipur,Central
O00006,Customer_023,Ahmedabad,West
O00007,Customer_036,Raipur,Central
O00008,Customer_019,Chennai,South
O00009,Customer_081,Mumbai,West
O00010,Customer_115,Surat,West


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 20: Join orders to products and calculate revenue by product category (PySpark)
#
# Explanation:
# 1. orders_parquet.fillna({"discount_pct": 0.0}): Replaces NULL discounts with 0.0 (same as COALESCE in SQL).
# 2. .withColumn("revenue", ...): Creates a reusable 'revenue' column using quantity * unit_price * (1 - discount_pct).
# 3. .join(products_parquet, "product_id", "inner"): Joins orders with products on product_id.
# 4. .groupBy("category").agg(...): Sums the revenue for each product category and rounds to 2 decimal places.

from pyspark.sql import functions as F

orders_clean = orders_parquet.fillna({"discount_pct": 0.0})
orders_clean = orders_clean.withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price") * (1 - F.col("discount_pct"))
)

orders_with_products = orders_clean.join(products_parquet, "product_id", "inner")

display(
    orders_with_products.groupBy("category")
    .agg(F.round(F.sum("revenue"), 2).alias("revenue"))
    .orderBy(F.desc("revenue"))
)

category,revenue
Electronics,1.425284365E7
Accessories,1.186723902E7
Furniture,1.136920708E7
Stationery,9317738.86


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 21: Find customers who have never placed an order (PySpark)
#
# Explanation:
# A "left_anti" join returns ONLY the rows from the left table (customers_parquet)
# that have NO matching customer_id in the right table (orders_parquet).

customers_without_orders = customers_parquet.join(
    orders_parquet.select("customer_id").distinct(),
    "customer_id",
    "left_anti"
)

display(customers_without_orders)

customer_id,customer_name,email,city,region,signup_date,customer_segment


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 22: Identify duplicate order IDs (PySpark)
#
# Explanation:
# 1. .groupBy("order_id").agg(F.count("*").alias("occurrences")): Counts how many times each order_id appears.
# 2. .filter(F.col("occurrences") > 1): Keeps only order_ids that appear more than once (duplicates).
#    (If the result is empty / 0 rows, it confirms there are no duplicate order_ids!)

duplicate_orders = (
    orders_parquet.groupBy("order_id")
    .agg(F.count("*").alias("occurrences"))
    .filter(F.col("occurrences") > 1)
)

display(duplicate_orders)

order_id,occurrences


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 23: Replace missing discounts with 0 and verify there are no NULL discounts (PySpark)
#
# Explanation:
# 1. .fillna({"discount_pct": 0.0}): Replaces all NULL values in discount_pct with 0.0.
# 2. .filter(F.col("discount_pct").isNull()).count(): Counts how many NULL values remain
#    after replacement (should return 0).

orders_no_null_discount = orders_parquet.fillna({"discount_pct": 0.0})

null_discount_count = (
    orders_no_null_discount
    .filter(F.col("discount_pct").isNull())
    .select(F.count("*").alias("null_discount_count"))
)

display(null_discount_count)

null_discount_count
0


In [0]:
# Section 7 - Level 3: Joins and data quality
# Question 24: Calculate the percentage of orders in each status (PySpark)
#
# Explanation:
# 1. total_orders = orders_parquet.count(): Gets the total number of orders (1,500).
# 2. .groupBy("order_status").agg(F.count("*").alias("order_count")): Counts orders per status.
# 3. .withColumn("pct_of_orders", F.round(100.0 * F.col("order_count") / total_orders, 2)):
#    Divides each status count by the total orders, multiplies by 100, and rounds to 2 decimals.

total_orders = orders_parquet.count()

orders_status_pct = (
    orders_parquet.groupBy("order_status")
    .agg(F.count("*").alias("order_count"))
    .withColumn("pct_of_orders", F.round(100.0 * F.col("order_count") / total_orders, 2))
    .orderBy(F.desc("pct_of_orders"))
)

display(orders_status_pct)

order_status,order_count,pct_of_orders
Delivered,938,62.53
Shipped,228,15.2
Pending,138,9.2
Cancelled,114,7.6
Returned,82,5.47


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 25: Rank products by revenue within each category (PySpark)
#
# What is a Window?
# Unlike groupBy() (which collapses rows into a single summary row per category),
# a Window lets us keep every individual product row while calculating a rank inside its category:
# 1. Window.partitionBy("category"): Creates a separate "window" (mini-group) for each category
#    so the ranking resets to #1 for every category (Electronics, Furniture, etc.).
# 2. .orderBy(F.desc("revenue")): Orders products inside each category from highest to lowest revenue.
# 3. F.dense_rank().over(w_category): Assigns rank 1, 2, 3... to each product within its category window.

from pyspark.sql.window import Window

product_revenue = (
    orders_clean.join(products_parquet, "product_id", "inner")
    .groupBy("category", "product_id", "product_name")
    .agg(F.round(F.sum("revenue"), 2).alias("revenue"))
)

w_category = Window.partitionBy("category").orderBy(F.desc("revenue"))

ranked_products = (
    product_revenue
    .withColumn("category_rank", F.dense_rank().over(w_category))
    .orderBy("category", "category_rank")
)

display(ranked_products)

category,product_id,product_name,revenue,category_rank
Accessories,P021,Power Bank,4004911.12,1
Accessories,P020,Desk Lamp,3310693.15,2
Accessories,P018,Phone Stand,2704208.09,3
Accessories,P017,Laptop Bag,1492429.08,4
Accessories,P019,Cable Organizer,354997.58,5
Electronics,P002,Keyboard,3449236.72,1
Electronics,P003,Monitor,3311863.41,2
Electronics,P001,Wireless Mouse,2310520.95,3
Electronics,P006,USB Hub,1991543.42,4
Electronics,P005,Webcam,1965297.16,5


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 26: For each customer, find their most recent order using ROW_NUMBER (PySpark)
#
# Explanation:
# 1. Window.partitionBy("customer_id"): Creates a separate window for each customer's orders.
# 2. .orderBy(F.to_date("order_date").desc(), F.col("order_id").desc()):
#    Sorts each customer's orders from newest date to oldest date (and breaks ties by order_id).
# 3. F.row_number().over(w_customer): Assigns #1 to the most recent order for each customer, #2 to the next, etc.
# 4. .filter("rn = 1"): Keeps ONLY row #1 (the single most recent order) for every customer.

w_customer = Window.partitionBy("customer_id").orderBy(
    F.to_date("order_date").desc(),
    F.col("order_id").desc()
)

latest_orders = (
    orders_parquet
    .withColumn("rn", F.row_number().over(w_customer))
    .filter("rn = 1")
)

display(latest_orders)

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method,rn
O00106,2026-01-29,C0001,P003,3,10031.39,0.0,Shipped,2026-02-02,Credit Card,1
O00894,2026-10-27,C0002,P008,7,5172.7,0.05,Shipped,2026-10-28,Debit Card,1
O00297,2026-08-20,C0003,P019,5,1714.55,0.2,Delivered,2026-08-24,Debit Card,1
O00500,2026-10-24,C0004,P006,2,7442.24,0.0,Delivered,2026-10-28,Credit Card,1
O00930,2026-10-24,C0005,P018,2,11400.54,0.0,Delivered,2026-10-29,Cash on Delivery,1
O00610,2026-07-22,C0006,P006,1,7442.24,0.0,Delivered,2026-07-25,Cash on Delivery,1
O00128,2026-09-20,C0007,P006,5,7442.24,0.0,Shipped,2026-09-27,Cash on Delivery,1
O00184,2026-06-27,C0008,P009,4,11965.21,0.15,Delivered,2026-06-28,Credit Card,1
O00865,2026-10-31,C0009,P015,5,9093.65,0.1,Delivered,2026-11-01,Cash on Delivery,1
O00142,2026-05-02,C0010,P019,1,1714.55,0.15,Delivered,null,Debit Card,1


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 27: Create a Gold table containing monthly revenue by region (PySpark)
#
# Explanation:
# 1. Joins orders_clean with customers_parquet on customer_id.
# 2. Extracts the year-month ('yyyy-MM') from order_date into 'order_month'.
# 3. Groups by order_month and region, summing and rounding the revenue to 2 decimal places.
# 4. Saves it as a Gold table in Unity Catalog (workspace.retail_lab.monthly_revenue_by_region).

monthly_gold = (
    orders_clean.join(customers_parquet, "customer_id", "inner")
    .withColumn("order_month", F.date_format(F.to_date("order_date"), "yyyy-MM"))
    .groupBy("order_month", "region")
    .agg(F.round(F.sum("revenue"), 2).alias("revenue"))
    .orderBy("order_month", "region")
)

monthly_gold.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.monthly_revenue_by_region")

display(monthly_gold)

order_month,region,revenue
2024-01,Central,417691.92
2024-01,East,179604.55
2024-01,North,238723.65
2024-01,South,26797.88
2024-01,West,345853.92
2024-02,Central,185762.55
2024-02,East,309875.45
2024-02,North,140640.51
2024-02,South,243304.09
2024-02,West,140588.8


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 28: Save the Gold table as Parquet and read it back (PySpark)
#
# Explanation:
# 1. monthly_gold.write.mode("overwrite").parquet(...): Writes the Gold DataFrame as Parquet files
#    into the /Volumes/workspace/retail_lab/parquet/monthly_revenue_by_region folder.
# 2. spark.read.parquet(...): Reads those Parquet files back into a new DataFrame and displays it.

gold_parquet_path = f"{parquet_base}/monthly_revenue_by_region"

monthly_gold.write.mode("overwrite").parquet(gold_parquet_path)

monthly_gold_readback = spark.read.parquet(gold_parquet_path)
display(monthly_gold_readback)

order_month,region,revenue
2024-01,Central,417691.92
2024-01,East,179604.55
2024-01,North,238723.65
2024-01,South,26797.88
2024-01,West,345853.92
2024-02,Central,185762.55
2024-02,East,309875.45
2024-02,North,140640.51
2024-02,South,243304.09
2024-02,West,140588.8


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 29 (Part 1): Save cleaned orders as a Delta table (PySpark)
#
# Explanation:
# .write.format("delta").mode("overwrite").saveAsTable(...) saves the DataFrame
# as a managed Delta table in Unity Catalog under workspace.retail_lab.cleaned_orders.

orders_clean.write.format("delta").mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.cleaned_orders"
)

In [0]:
%sql
-- Section 7 - Level 4: Window functions and storage
-- Question 29 (Part 2): Query the cleaned_orders Delta table with SQL
SELECT * FROM workspace.retail_lab.cleaned_orders LIMIT 10;

order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,order_status,shipping_date,payment_method,revenue
O00001,2025-04-18,C0032,P017,7,6951.23,0.0,Cancelled,null,UPI,48658.61
O00002,2026-03-17,C0089,P006,1,7442.24,0.0,Delivered,2026-03-21,Cash on Delivery,7442.24
O00003,2026-07-23,C0087,P007,6,4325.82,0.1,Pending,null,UPI,23359.428
O00004,2024-01-21,C0092,P011,6,7404.01,0.1,Delivered,2024-01-25,UPI,39981.654
O00005,2024-04-08,C0078,P014,1,8394.51,0.15,Pending,null,Cash on Delivery,7135.3335
O00006,2025-03-06,C0023,P007,5,4325.82,0.0,Delivered,2025-03-11,UPI,21629.1
O00007,2024-04-17,C0036,P020,7,17319.87,0.1,Delivered,2024-04-21,Net Banking,109115.181
O00008,2024-12-25,C0019,P021,3,12639.77,0.05,Shipped,2024-12-29,Debit Card,36023.3445
O00009,2026-03-21,C0081,P011,4,7404.01,0.2,Delivered,2026-03-23,Net Banking,23692.832000000002
O00010,2025-08-14,C0115,P007,4,4325.82,0.1,Delivered,null,Net Banking,15572.952


In [0]:
# Section 7 - Level 4: Window functions and storage
# Question 30: Explain what additional capabilities Delta provides compared with Parquet
#
# What is Delta (Delta Lake)?
# Delta is the default storage format in Databricks.
# Formula: Delta = Parquet files + a Transaction Log (_delta_log)
#
# The problem with plain Parquet:
# Plain Parquet is just a static file format. Once written, you cannot easily edit it:
# - To update 1 row or delete 1 order, you have to rewrite the entire dataset.
# - If a job crashes halfway through writing, it leaves corrupted/half-written files behind.
# - You cannot see previous versions of the data.
#
##ANSWER:
# How Delta solves this (Additional capabilities over Parquet):

# Under the hood, Delta still stores data as Parquet files, but adds a '_delta_log' folder
# (a JSON log/diary of every change made to the table) so it acts like a real database table:
# 1. Edit Rows Directly (UPDATE, DELETE, MERGE): Update or delete specific records without
#    rewriting the entire dataset manually.
# 2. ACID Transactions (No Corrupted Data): If a write fails at 99%, Delta rolls it back so
#    the table is never left half-broken, and multiple users can read/write safely at the same time.
# 3. Time Travel (Data Versioning): Query or restore what the table looked like at an earlier
#    version or timestamp (e.g., VERSION AS OF or TIMESTAMP AS OF) and view DESCRIBE HISTORY.
# 4. Schema Enforcement & Evolution: Automatically blocks bad data (like text in a price column)
#    while allowing safe schema updates when needed.
# 5. Faster Queries (Data Skipping): Uses statistics stored in the transaction log to skip
#    irrelevant Parquet files during queries.